In [1]:
import os, sys
import pandas as pd
import numpy as np

cwd = os.getcwd()

file_CO = 'FR001400AJ45_20231027_CO.parquet.gzip'
CO = pd.read_parquet(file_CO)

file_FO = 'FR0000120628_20231027_FO.parquet.gzip'
FO = pd.read_parquet(file_FO)

file_TIF = 'FR0000051807_20231005_TIF.parquet.gzip'
TIF = pd.read_parquet(file_TIF)

In [2]:
df = CO.copy()
df = df.between_time('9:00', '17:35').reset_index().groupby(['event_time_cet','order_side']).last()
display(df)
df = df.unstack()
df.columns = ['{}_CO_size'.format(side.lower()) for _, side in df.columns]
display(df)

order_size
event_time_cet      order_side            
2023-10-27 09:00:00 Buy           811946.0
                    Sell         1004046.0
2023-10-27 09:05:00 Buy           726805.0
                    Sell          863445.0
2023-10-27 09:10:00 Buy           590684.0
...                                    ...
2023-10-27 17:25:00 Sell          294759.0
2023-10-27 17:30:00 Buy           392002.0
                    Sell          196644.0
2023-10-27 17:35:00 Buy           778799.0
                    Sell         1680448.0

[208 rows x 1 columns]

,buy_CO_size,sell_CO_size
event_time_cet,,
2023-10-27 09:00:00,811946.0,1004046.0
2023-10-27 09:05:00,726805.0,863445.0
2023-10-27 09:10:00,590684.0,626471.0
2023-10-27 09:15:00,717560.0,759915.0
2023-10-27 09:20:00,765040.0,739836.0
...,...,...
2023-10-27 17:15:00,401796.0,412086.0
2023-10-27 17:20:00,253029.0,180892.0
2023-10-27 17:25:00,314971.0,294759.0


In [7]:
df['time'] = df.index
df['time'] = (df['time'].dt.hour * 60 + df['time'].dt.minute)
display(df.reset_index())

,event_time_cet,buy_CO_size,sell_CO_size,time
0,2023-10-27 09:00:00,811946.0,1004046.0,540
1,2023-10-27 09:05:00,726805.0,863445.0,545
2,2023-10-27 09:10:00,590684.0,626471.0,550
3,2023-10-27 09:15:00,717560.0,759915.0,555
4,2023-10-27 09:20:00,765040.0,739836.0,560
...,...,...,...,...
99,2023-10-27 17:15:00,401796.0,412086.0,1035
100,2023-10-27 17:20:00,253029.0,180892.0,1040
101,2023-10-27 17:25:00,314971.0,294759.0,1045
102,2023-10-27 17:30:00,392002.0,196644.0,1050


In [12]:
df = FO.copy()
display(df)
n = 5

df = df[df['order_side'] == 'Buy'].drop(['order_side'], axis=1)
df = df.between_time('9:00', '17:35').reset_index().groupby(['event_time_cet'], as_index=False).apply(lambda x: x.nlargest(n, 'trade_size'))
df = df.set_index(['event_time_cet'])

tmp = df.index.value_counts().to_frame()
tmp['count'] = n - tmp['count']
tmp = tmp[tmp['count'] != 0]
ls = []
for idx, c in tmp.iterrows():
    ls.insert(c['count'], idx)

tmp = pd.DataFrame(index=ls)
tmp.index.name = df.index.name
df = pd.concat([df,tmp]).reset_index().fillna(0)
df['idx'] = df.groupby('event_time_cet')['trade_size'].rank(ascending=False, method='first', na_option='top')
df = df.pivot(index=['event_time_cet'], columns=['idx'], values=['trade_price', 'trade_size']).T.reset_index()
df = df.groupby(['idx','level_0']).last().T.fillna(0)
df.columns = ['FO_{}_{}'.format(data_t.split('_')[1], int(rank)) for rank, data_t in df.columns]
display(df)
'''
df = df.reset_index(drop=True)
df = df.groupby(['event_time_cet','trade_price'], as_index=True).last()
display(df)
display(df.unstack())
#df.columns = df.columns.swaplevel(0, 1)
df.index = df.index.droplevel(1)
df = df.sort_index(axis=1)
display(df)'''

,order_side,trade_price,trade_size
event_time_cet,,,
2023-10-27 09:00:00,Buy,27.610,477.0
2023-10-27 09:00:00,Buy,27.620,905.0
2023-10-27 09:00:00,Buy,27.625,419.0
2023-10-27 09:00:00,Buy,27.630,452.0
2023-10-27 09:00:00,Buy,27.650,402.0
...,...,...,...
2023-10-27 17:25:00,Sell,27.520,1880.0
2023-10-27 17:25:00,Sell,27.525,7307.0
2023-10-27 17:25:00,Sell,27.530,3564.0


,FO_price_1,FO_size_1,FO_price_2,FO_size_2,FO_price_3,FO_size_3,FO_price_4,FO_size_4,FO_price_5,FO_size_5
event_time_cet,,,,,,,,,,
2023-10-27 09:00:00,27.720,104666.0,27.690,3056.0,27.680,1510.0,27.675,1010.0,27.620,905.0
2023-10-27 09:05:00,27.700,7015.0,27.670,5430.0,27.645,4190.0,27.655,1929.0,27.660,1736.0
2023-10-27 09:10:00,27.615,4190.0,27.590,2896.0,27.625,1206.0,27.560,874.0,27.585,836.0
2023-10-27 09:15:00,27.570,2951.0,27.565,2063.0,27.550,928.0,27.560,746.0,27.575,730.0
2023-10-27 09:20:00,27.560,1180.0,27.565,816.0,27.570,550.0,27.545,237.0,27.575,201.0
...,...,...,...,...,...,...,...,...,...,...
2023-10-27 17:10:00,27.535,2801.0,27.510,2699.0,27.500,1636.0,27.505,1528.0,27.530,1172.0
2023-10-27 17:15:00,27.565,6469.0,27.550,3278.0,27.540,2251.0,27.555,2010.0,27.560,1928.0
2023-10-27 17:20:00,27.530,10225.0,27.525,9323.0,27.535,8985.0,27.520,500.0,27.540,48.0


"\ndf = df.reset_index(drop=True)\ndf = df.groupby(['event_time_cet','trade_price'], as_index=True).last()\ndisplay(df)\ndisplay(df.unstack())\n#df.columns = df.columns.swaplevel(0, 1)\ndf.index = df.index.droplevel(1)\ndf = df.sort_index(axis=1)\ndisplay(df)"

In [14]:
df = TIF.copy()
display(df)
df = df.between_time('9:00', '17:35').reset_index().groupby(['index','side','order_type','time_in_force']).last()
df = df.unstack(['side','order_type','time_in_force'])
df.columns = ['{}_{}_{}_{}'.format(side.lower(), o_type, tif, size) for size, side, o_type, tif in df.columns]
display(df)

,side,order_type,time_in_force,size
index,,,,
2023-10-05 08:20:00,Buy,Limit,Valid for Closing,0.0
2023-10-05 08:20:00,Buy,Limit,Valid for Uncrossing,0.0
2023-10-05 08:20:00,Buy,Market,Valid for Closing,0.0
2023-10-05 08:20:00,Buy,Market,Valid for Uncrossing,9.0
2023-10-05 08:20:00,Sell,Limit,Valid for Closing,0.0
...,...,...,...,...
2023-10-05 17:35:00,Buy,Market,Valid for Uncrossing,0.0
2023-10-05 17:35:00,Sell,Limit,Valid for Closing,0.0
2023-10-05 17:35:00,Sell,Limit,Valid for Uncrossing,0.0


,buy_Limit_Valid for Closing_size,buy_Limit_Valid for Uncrossing_size,buy_Market_Valid for Closing_size,buy_Market_Valid for Uncrossing_size,sell_Limit_Valid for Closing_size,sell_Limit_Valid for Uncrossing_size,sell_Market_Valid for Closing_size,sell_Market_Valid for Uncrossing_size
index,,,,,,,,
2023-10-05 09:00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2023-10-05 09:15:00,0.0,0.0,0.0,0.0,0.0,0.0,22.0,0.0
2023-10-05 09:20:00,0.0,0.0,0.0,0.0,0.0,0.0,22.0,0.0
2023-10-05 09:25:00,0.0,0.0,0.0,0.0,0.0,0.0,22.0,0.0
2023-10-05 09:30:00,0.0,0.0,0.0,0.0,0.0,0.0,22.0,0.0
...,...,...,...,...,...,...,...,...
2023-10-05 17:15:00,0.0,0.0,12669.0,0.0,0.0,0.0,162.0,0.0
2023-10-05 17:20:00,0.0,0.0,12669.0,0.0,0.0,0.0,162.0,0.0
2023-10-05 17:25:00,0.0,0.0,12869.0,0.0,0.0,0.0,1345.0,0.0


In [11]:
n = 5
DF = pd.DataFrame()
for s in ['Buy','Sell']:
    df = FO.copy()
    df = df[df['order_side'] == s].drop(['order_side'], axis=1)
    df = df.between_time('9:00', '17:35').reset_index().groupby(['event_time_cet'], as_index=False).apply(lambda x: x.nlargest(n, 'trade_size'))
    df = df.set_index(['event_time_cet'])

    tmp = df.index.value_counts().to_frame()
    tmp['count'] = n - tmp['count']
    tmp = tmp[tmp['count'] != 0]
    ls = []
    for idx, c in tmp.iterrows():
        ls.insert(c['count'], idx)

    tmp = pd.DataFrame(index=ls)
    tmp.index.name = df.index.name
    df = pd.concat([df,tmp]).reset_index().fillna(0)
    df['idx'] = df.groupby('event_time_cet')['trade_size'].rank(ascending=False, method='first', na_option='top')
    df = df.pivot(index=['event_time_cet'], columns=['idx'], values=['trade_price', 'trade_size']).T.reset_index()
    df = df.groupby(['idx','level_0']).last().T.fillna(0)
    df.columns = ['{}_FO_{}_{}'.format(s, data_t.split('_')[1], int(rank)) for rank, data_t in df.columns]
    DF = pd.concat([DF, df], ignore_index=False, axis=1)
display(DF)

,Buy_FO_price_1,Buy_FO_size_1,Buy_FO_price_2,Buy_FO_size_2,Buy_FO_price_3,Buy_FO_size_3,Buy_FO_price_4,Buy_FO_size_4,Buy_FO_price_5,Buy_FO_size_5,Sell_FO_price_1,Sell_FO_size_1,Sell_FO_price_2,Sell_FO_size_2,Sell_FO_price_3,Sell_FO_size_3,Sell_FO_price_4,Sell_FO_size_4,Sell_FO_price_5,Sell_FO_size_5
event_time_cet,,,,,,,,,,,,,,,,,,,,
2023-10-27 09:00:00,27.720,104666.0,27.690,3056.0,27.680,1510.0,27.675,1010.0,27.620,905.0,27.720,103912.0,27.690,3056.0,27.680,1510.0,27.675,1010.0,27.620,905.0
2023-10-27 09:05:00,27.700,7015.0,27.670,5430.0,27.645,4190.0,27.655,1929.0,27.660,1736.0,27.700,7015.0,27.670,5430.0,27.645,4190.0,27.655,1929.0,27.660,1736.0
2023-10-27 09:10:00,27.615,4190.0,27.590,2896.0,27.625,1206.0,27.560,874.0,27.585,836.0,27.615,4190.0,27.590,2896.0,27.625,1206.0,27.560,874.0,27.585,836.0
2023-10-27 09:15:00,27.570,2951.0,27.565,2063.0,27.550,928.0,27.560,746.0,27.575,730.0,27.570,2951.0,27.565,2063.0,27.550,928.0,27.560,746.0,27.575,730.0
2023-10-27 09:20:00,27.560,1180.0,27.565,816.0,27.570,550.0,27.545,237.0,27.575,201.0,27.560,1180.0,27.565,816.0,27.570,550.0,27.545,237.0,27.575,201.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2023-10-27 17:10:00,27.535,2801.0,27.510,2699.0,27.500,1636.0,27.505,1528.0,27.530,1172.0,27.535,2801.0,27.510,2699.0,27.500,1636.0,27.505,1528.0,27.530,1172.0
2023-10-27 17:15:00,27.565,6469.0,27.550,3278.0,27.540,2251.0,27.555,2010.0,27.560,1928.0,27.565,6469.0,27.550,3278.0,27.540,2251.0,27.555,2010.0,27.560,1928.0
2023-10-27 17:20:00,27.530,10225.0,27.525,9323.0,27.535,8985.0,27.520,500.0,27.540,48.0,27.530,10225.0,27.525,9323.0,27.535,8985.0,27.520,500.0,27.540,48.0
